# Notebook técnico del TFM — versión reproducible y verificada

Pipeline completo para el análisis de la relación entre el discurso institucional del
Ministerio de Economía (`@_minecogob`) y el Índice de Confianza del Consumidor (CCI)
de Eurostat en España, periodo **2024–2026**.

**Nota de reproducibilidad:** la extracción original mediante la API oficial de X/Twitter
fue ejecutada previamente y generó el archivo `tweets_minecogob.csv`.
Al reejecutar el notebook, la sección de extracción puede omitirse (`RUN_EXTRACTION = False`)
si se dispone del CSV original. El análisis completo (preprocesamiento → sentimiento →
estadística) es totalmente reproducible desde ese archivo.

**Pipeline:**
1. Extracción desde la API de X/Twitter *(opcional — controlado por `RUN_EXTRACTION`)*
2. Carga y preparación del CCI (Eurostat)
3. Carga e inspección del corpus de tweets
4. Preprocesamiento y limpieza textual
5. Análisis de sentimiento con `pysentimiento` (RoBERTa-es) — validación del modelo
6. Clasificación del corpus completo (716 tweets)
7. Construcción de la serie temporal mensual (`resample('M')`)
8. Fusión de series y visualización comparativa
9. Estadísticas descriptivas del corpus
10. Nube de palabras del discurso institucional
11. Análisis estadístico: correlaciones (Pearson, Spearman) y estacionariedad (ADF)
12. Causalidad de Granger y CCF sobre series en nivel
13. Análisis de sensibilidad: series diferenciadas (Δ)
14. Distribución de clases de sentimiento

---

## 1. Configuración de la API de X/Twitter *(ejecución opcional)*

El Bearer Token se lee desde la variable de entorno `X_BEARER_TOKEN` — **nunca se
almacena en el notebook** para evitar exposición accidental de credenciales.

La extracción está desactivada por defecto (`RUN_EXTRACTION = False`).
Cambiar a `True` únicamente si se dispone de token activo y créditos API disponibles.

Correcciones aplicadas respecto a versiones anteriores:
- `start_time` fijado en **2024-01-01** (el estudio cubre 2024–2026).
- Parámetro `"exclude": "retweets"` incluido en la llamada a la API como primera
  línea de defensa contra retweets (complementada por filtro pandas en preprocesamiento).
- Manejo explícito de errores HTTP con `raise_for_status()`.

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path

# ── Configuración ─────────────────────────────────────────────────────────────
BEARER_TOKEN    = os.getenv("X_BEARER_TOKEN")   # Nunca hardcodear en el notebook
MINECOGOB_ID    = "494041400"                    # ID numérico estable de @_minecogob
RUN_EXTRACTION  = False   # Cambiar a True solo si hay token activo y créditos disponibles


def get_headers():
    if not BEARER_TOKEN:
        raise RuntimeError(
            "No se encontró X_BEARER_TOKEN. "
            "Define el token como variable de entorno o mantén RUN_EXTRACTION=False."
        )
    return {"Authorization": f"Bearer {BEARER_TOKEN}"}


def get_tweets(user_id, max_results=100, next_token=None):
    """Obtiene tweets originales del usuario (sin retweets) vía API v2."""
    url    = f"https://api.twitter.com/2/users/{user_id}/tweets"
    params = {
        "max_results":  max_results,
        "tweet.fields": "created_at,text,public_metrics",
        "start_time":   "2024-01-01T00:00:00Z",
        "end_time":     "2026-03-31T23:59:59Z",
        "exclude":      "retweets",        # Primera línea de defensa
    }
    if next_token:
        params["pagination_token"] = next_token
    r = requests.get(url, headers=get_headers(), params=params)
    r.raise_for_status()
    return r.json()


# ── Verificación de conectividad y extracción completa ────────────────────────
if RUN_EXTRACTION:
    url_check = f"https://api.twitter.com/2/users/{MINECOGOB_ID}"
    r_check   = requests.get(url_check, headers=get_headers())
    print(f"Estado API: {r_check.status_code} — "
          f"Usuario: {r_check.json().get('data', {}).get('name', 'N/A')}")

    all_tweets = []
    next_token = None

    for i in range(20):   # máximo 20 páginas × 100 = 2 000 tweets
        data = get_tweets(MINECOGOB_ID, next_token=next_token)

        if "data" not in data:
            print(f"Fin en página {i+1}:", data)
            break

        all_tweets.extend(data["data"])
        print(f"Página {i+1}: {len(data['data'])} tweets — "
              f"Total acumulado: {len(all_tweets)}")

        if "meta" in data and "next_token" in data["meta"]:
            next_token = data["meta"]["next_token"]
        else:
            print("No hay más páginas.")
            break

    df_raw = pd.DataFrame(all_tweets)
    df_raw.to_csv("tweets_minecogob.csv", index=False)
    print(f"\n✅ Total guardado: {len(df_raw)} tweets → tweets_minecogob.csv")
    print(f"   Rango temporal: {df_raw['created_at'].min()} → {df_raw['created_at'].max()}")

else:
    print("Extracción omitida (RUN_EXTRACTION=False). "
          "Continuar con tweets_minecogob.csv generado previamente.")

## 2. Carga y preparación del CCI — Eurostat

Se importa `ei_bsco_m_linear.csv`, descarga directa del portal de Eurostat con el
**Consumer Confidence Indicator (CCI)** mensual.

Pasos aplicados:
- Filtrado para **España** e indicador de confianza del consumidor.
- Acotación del periodo: **enero 2024 – marzo 2026**.
- Selección de la variante **ajustada estacionalmente** (`Seasonally adjusted`),
  que elimina efectos cíclicos periódicos y permite comparaciones limpias entre meses.

El resultado se exporta como `cci_spain.csv`.

In [ ]:
from pathlib import Path
import pandas as pd

csv_cci = Path("ei_bsco_m_linear.csv")
if not csv_cci.exists():
    raise FileNotFoundError(
        "No se encontró ei_bsco_m_linear.csv. "
        "Descarga la serie CCI de Eurostat y colócala en la misma carpeta del notebook."
    )

cci = pd.read_csv(csv_cci)
print(f"Shape total: {cci.shape}")
print(f"Columnas   : {cci.columns.tolist()}")
print(cci.head(3))

# ── Filtrar España + Consumer Confidence Indicator ────────────────────────────
cci_es = cci[
    (cci['geo']   == 'Spain') &
    (cci['indic'] == 'Consumer confidence indicator')
].copy()

# ── Convertir fecha y acotar periodo 2024–2026 ────────────────────────────────
cci_es['TIME_PERIOD'] = pd.to_datetime(cci_es['TIME_PERIOD'])
cci_es = cci_es[
    (cci_es['TIME_PERIOD'] >= '2024-01-01') &
    (cci_es['TIME_PERIOD'] <= '2026-03-31')
].copy()

# ── Seleccionar únicamente la serie ajustada estacionalmente ─────────────────
cci_final = cci_es[
    cci_es['s_adj'].str.startswith('Seasonally')
][['TIME_PERIOD', 'OBS_VALUE']].copy().reset_index(drop=True)
cci_final.columns = ['date', 'cci_value']

cci_final['date']       = pd.to_datetime(cci_final['date'])
cci_final['year_month'] = cci_final['date'].dt.to_period('M')

print(f"\nCCI España ajustado estacionalmente ({len(cci_final)} meses):")
print(cci_final)
cci_final.to_csv("cci_spain.csv", index=False)
print("\n✅ cci_spain.csv guardado correctamente")

## 3. Carga e inspección del corpus de tweets

Se carga `tweets_minecogob.csv` generado durante la extracción original.
Se verifica estructura, rango temporal y primeras entradas antes de procesar.

In [ ]:
from pathlib import Path
import pandas as pd

csv_tweets = Path("tweets_minecogob.csv")
if not csv_tweets.exists():
    raise FileNotFoundError(
        "No se encontró tweets_minecogob.csv. "
        "Coloca en la misma carpeta el CSV de la extracción original "
        "o activa RUN_EXTRACTION=True con token válido."
    )

df_raw = pd.read_csv(csv_tweets)
print(f"Shape             : {df_raw.shape}")
print(f"Columnas          : {df_raw.columns.tolist()}")
print(f"Fecha más antigua : {df_raw['created_at'].min()}")
print(f"Fecha más reciente: {df_raw['created_at'].max()}")
print("\nPrimeras entradas:")
print(df_raw.head(3))

## 4. Preprocesamiento y limpieza textual del corpus

Esta sección implementa el pipeline completo de limpieza y separación del corpus.

**Extracción de métricas:** el campo `public_metrics` (almacenado como string JSON)
se parsea para obtener `retweet_count`, `like_count` y `reply_count` como columnas numéricas.

**Filtro de retweets (segunda línea de defensa):** aunque la API ya los excluye con
`"exclude": "retweets"`, se aplica un filtro adicional con pandas (`~str.startswith('RT @')`)
para garantizar la integridad del corpus en cualquier re-ejecución desde CSV.

**Verificación del corpus:** se comprueba que el total de tweets originales
coincide con los **716 documentados en la memoria del TFM**.

**Limpieza textual con Regex:**

| Operación | Patrón | Justificación |
|---|---|---|
| Eliminar URLs | `https?://\S+` | Ruido semántico sin valor para el modelo |
| Eliminar menciones | `@\w+` | No aportan al discurso económico institucional |
| Eliminar caracteres de control | `[\r\t]` | Normalización |
| Colapsar espacios múltiples | `\s+` | Normalización |
| **Mantener hashtags `#`** | — | RoBERTa-es puede interpretar su contexto semántico |

El corpus limpio se exporta como `tweets_minecogob_limpio.csv`.

In [ ]:
import pandas as pd
import ast
import re

# ── Cargar y parsear tweets ───────────────────────────────────────────────────
df = pd.read_csv("tweets_minecogob.csv")
df['created_at'] = pd.to_datetime(df['created_at'])
df['year_month'] = df['created_at'].dt.to_period('M')

# Parsear public_metrics (string → dict)
df['public_metrics'] = df['public_metrics'].apply(ast.literal_eval)
df['retweet_count']  = df['public_metrics'].apply(lambda x: x['retweet_count'])
df['like_count']     = df['public_metrics'].apply(lambda x: x['like_count'])
df['reply_count']    = df['public_metrics'].apply(lambda x: x['reply_count'])

# ── Separar retweets (doble defensa tras el filtro de API) ────────────────────
df_original = df[~df['text'].str.startswith('RT @')].copy()
df_rt       = df[ df['text'].str.startswith('RT @')].copy()
print(f"Tweets originales del Ministerio : {len(df_original)}")
print(f"Retweets descartados             : {len(df_rt)}")

# ── Verificación del corpus documentado en la memoria del TFM ─────────────────
assert len(df_original) == 716, (
    f"⚠️  Se esperaban 716 tweets originales pero se encontraron {len(df_original)}. "
    "Revisar parámetros de extracción o el CSV fuente."
)
print("✅ Corpus verificado: 716 tweets originales")

# ── Función de limpieza textual con Regex ─────────────────────────────────────
def limpiar_texto(texto: str) -> str:
    """
    Limpieza de tweets para análisis de sentimiento (PLN).
    - Elimina URLs        → ruido semántico.
    - Elimina @menciones  → no aportan al discurso económico.
    - Mantiene #hashtags  → RoBERTa-es puede interpretar su contexto.
    - Normaliza espacios y caracteres de control.
    """
    texto = re.sub(r'https?://\S+', '', texto)   # URLs
    texto = re.sub(r'@\w+',         '', texto)   # Menciones @usuario
    texto = re.sub(r'[\r\t]',       ' ', texto)  # Caracteres de control
    texto = re.sub(r'\s+',           ' ', texto).strip()  # Espacios múltiples
    return texto

# ── Aplicar limpieza ──────────────────────────────────────────────────────────
df_original['text_clean'] = df_original['text'].apply(limpiar_texto)

# Muestra de verificación
print("\n── Muestra de limpieza (original → limpio) ──────────────────────────────")
for _, row in df_original.head(3).iterrows():
    print(f"  ORIG : {row['text'][:110]}")
    print(f"  CLEAN: {row['text_clean'][:110]}")
    print()

# ── Exportar corpus limpio ────────────────────────────────────────────────────
df_original.to_csv("tweets_minecogob_limpio.csv", index=False)
print("✅ tweets_minecogob_limpio.csv guardado con columna text_clean")

## 5. Análisis de sentimiento con `pysentimiento` (RoBERTa-es) — validación del modelo

Se utiliza `pysentimiento`, librería especializada en PLN para español basada en
modelos transformadores (RoBERTa) entrenados sobre corpus de redes sociales.

El analizador se inicializa explícitamente para **español** (`lang="es"`).
Se realiza una prueba de validación con un tweet real antes del procesamiento masivo.

El modelo devuelve:
- **Etiqueta ganadora**: `POS`, `NEG` o `NEU`.
- **Probabilidades continuas** para cada clase: la base del *sentiment score* ponderado.

In [ ]:
!pip install pysentimiento --quiet

from pysentimiento import create_analyzer

# ✅ Instanciación explícita en español
analyzer = create_analyzer(task="sentiment", lang="es")

# ── Prueba de validación con tweet real del corpus ────────────────────────────
tweet_prueba = "La economía española confirma su aceleración en el 4T2025, creciendo un 0,8%"
resultado    = analyzer.predict(tweet_prueba)
print(f"Tweet     : {tweet_prueba}")
print(f"Etiqueta  : {resultado.output}")
print(f"Prob. POS : {resultado.probas['POS']:.4f}")
print(f"Prob. NEG : {resultado.probas['NEG']:.4f}")
print(f"Prob. NEU : {resultado.probas['NEU']:.4f}")

## 6. Clasificación de sentimiento sobre el corpus completo (716 tweets)

Se aplica el modelo tweet a tweet sobre el corpus limpio.
Para cada tweet se almacenan la etiqueta ganadora y las tres probabilidades continuas.

El manejo de errores registra explícitamente el tweet fallido con su índice y texto
(sin asignación ciega de 0.33/0.33/0.33), lo que permite auditar anomalías.

El resultado se exporta como `tweets_con_sentimiento.csv`.

In [ ]:
import pandas as pd

df = pd.read_csv("tweets_minecogob_limpio.csv")
print(f"Tweets a analizar: {len(df)}")
print("Analizando sentimiento... (puede tardar 2–3 minutos)")

results = []
errores = []

for i, row in df.iterrows():
    try:
        pred = analyzer.predict(str(row['text_clean']))
        results.append({
            'sentiment': pred.output,
            'prob_pos':  pred.probas['POS'],
            'prob_neg':  pred.probas['NEG'],
            'prob_neu':  pred.probas['NEU'],
        })
    except Exception as e:
        errores.append({'index': i, 'texto': str(row['text_clean'])[:80], 'error': str(e)})
        results.append({'sentiment': 'NEU', 'prob_pos': 0.33, 'prob_neg': 0.33, 'prob_neu': 0.34})

    if (i + 1) % 100 == 0:
        print(f"  Procesados: {i+1}/{len(df)}")

df_sent = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
df_sent.to_csv("tweets_con_sentimiento.csv", index=False)

print(f"\n✅ Completado. Shape: {df_sent.shape}")
print(f"   Errores registrados: {len(errores)}")
if errores:
    print("   Detalle de errores:")
    for e in errores:
        print(f"     [{e['index']}] {e['texto']} → {e['error']}")
print("\nDistribución de etiquetas:")
print(df_sent['sentiment'].value_counts())

## 7. Construcción de la serie temporal mensual (`resample`)

Se agrupan los tweets a nivel mensual utilizando `df.resample('M')`, conforme a la
metodología descrita en la memoria del TFM (no `groupby`, que no opera sobre un
índice temporal real).

El índice del DataFrame se establece como `created_at` (`DatetimeIndex`) antes del
resampleo para garantizar la alineación temporal correcta.

Para cada mes se calculan:
- Número de tweets originales (`n_tweets`).
- Probabilidad media de cada clase: `mean_pos`, `mean_neg`, `mean_neu`.
- Porcentaje de tweets positivos y negativos.
- **Sentiment score neto**: `mean_pos − mean_neg` — indicador continuo y ponderado
  del tono institucional mensual, más informativo que una variable categórica.

In [ ]:
import pandas as pd

df_sent = pd.read_csv("tweets_con_sentimiento.csv")

# ✅ Establecer índice datetime para habilitar resample()
df_sent['created_at'] = pd.to_datetime(df_sent['created_at'])
df_sent = df_sent.set_index('created_at').sort_index()

# ✅ Resampleo mensual — conforme a metodología TFM
monthly = df_sent.resample('M').agg(
    n_tweets  = ('text',      'count'),
    mean_pos  = ('prob_pos',  'mean'),
    mean_neg  = ('prob_neg',  'mean'),
    mean_neu  = ('prob_neu',  'mean'),
    pct_pos   = ('sentiment', lambda x: (x == 'POS').mean()),
    pct_neg   = ('sentiment', lambda x: (x == 'NEG').mean()),
).reset_index()

monthly.rename(columns={'created_at': 'date'}, inplace=True)
monthly['year_month']      = monthly['date'].dt.to_period('M')
monthly['sentiment_score'] = monthly['mean_pos'] - monthly['mean_neg']

print(monthly[['year_month', 'n_tweets', 'sentiment_score', 'pct_pos', 'pct_neg']])
print(f"\nTotal tweets en serie mensual: {monthly['n_tweets'].sum()}")

## 8. Fusión de series y visualización comparativa: Sentiment score vs. CCI España

Se combinan ambas series temporales mediante `merge` por mes (`year_month`).
El gráfico de doble eje Y muestra la evolución paralela del tono institucional
y la confianza del consumidor en el periodo 2024–2026.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# ── Cargar CCI y fusionar ─────────────────────────────────────────────────────
cci_final = pd.read_csv("cci_spain.csv")
cci_final['year_month'] = pd.to_datetime(cci_final['date']).dt.to_period('M')

merged        = monthly.merge(cci_final, on='year_month', how='inner')
merged['date_ts'] = merged['year_month'].dt.to_timestamp()
print(f"Meses con ambas series disponibles: {len(merged)}")
print(f"Rango de fechas: {merged['year_month'].min()} a {merged['year_month'].max()}")
print(f"Total tweets analizados: {merged['n_tweets'].sum()}")
print("\nPrimeras filas:")
print(merged[['year_month', 'n_tweets', 'sentiment_score', 'cci_value']].head())
print("\nÚltimas filas:")
print(merged[['year_month', 'n_tweets', 'sentiment_score', 'cci_value']].tail())

# ── Gráfico de doble eje ──────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(14, 6))

color1 = '#2196F3'
ax1.set_xlabel('Mes')
ax1.set_ylabel('Sentiment score (discurso institucional)', color=color1)
ax1.plot(merged['date_ts'], merged['sentiment_score'],
         color=color1, linewidth=2, marker='o', markersize=4, label='Sentiment score')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(y=0, color=color1, linestyle='--', alpha=0.3)

ax2 = ax1.twinx()
color2 = '#F44336'
ax2.set_ylabel('CCI España (ajustado estacionalmente)', color=color2)
ax2.plot(merged['date_ts'], merged['cci_value'],
         color=color2, linewidth=2, marker='s', markersize=4, label='CCI España')
ax2.tick_params(axis='y', labelcolor=color2)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Sentiment score del discurso institucional vs. CCI España (2024–2026)')
plt.tight_layout()
plt.savefig('series_temporales.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado: series_temporales.png")

## 9. Estadísticas descriptivas del corpus

Análisis cuantitativo de la estructura de los tweets institucionales:
longitud de caracteres y prevalencia de elementos comunicativos
(menciones, hashtags, URLs, cifras, emojis).

In [ ]:
import pandas as pd
import re

df_sent = pd.read_csv("tweets_con_sentimiento.csv")

# ── Longitud de caracteres ────────────────────────────────────────────────────
df_sent['char_length'] = df_sent['text'].str.len()
print("=== ESTADÍSTICAS DE LONGITUD ===")
print(df_sent['char_length'].describe().round(1))

# ── Tipos de contenido ────────────────────────────────────────────────────────
df_sent['tiene_mencion'] = df_sent['text'].str.contains(r'@\w+',    regex=True)
df_sent['tiene_hashtag'] = df_sent['text'].str.contains(r'#\w+',    regex=True)
df_sent['tiene_url']     = df_sent['text'].str.contains(r'http',     regex=True)
df_sent['tiene_numero']  = df_sent['text'].str.contains(r'\d',       regex=True)
df_sent['tiene_emoji']   = df_sent['text'].str.contains(
    r'[\U0001F300-\U0001FAFF\u2600-\u27BF]', regex=True)

print("\n=== TIPOS DE CONTENIDO (% del corpus) ===")
print(f"Con mención (@)  : {df_sent['tiene_mencion'].mean()*100:.1f}%")
print(f"Con hashtag (#)  : {df_sent['tiene_hashtag'].mean()*100:.1f}%")
print(f"Con URL/enlace   : {df_sent['tiene_url'].mean()*100:.1f}%")
print(f"Con cifras/datos : {df_sent['tiene_numero'].mean()*100:.1f}%")
print(f"Con emoji        : {df_sent['tiene_emoji'].mean()*100:.1f}%")

## 10. Nube de palabras del discurso institucional

Visualización de los términos más frecuentes en el corpus de tweets originales.
Se eliminan stopwords en español y términos propios de la cuenta institucional
antes de generar la nube.

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re

# ── Histograma de longitud de caracteres ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_sent['char_length'], bins=20, color='#2196F3', edgecolor='white', alpha=0.85)
ax.axvline(df_sent['char_length'].mean(), color='#F44336', linestyle='--', linewidth=2,
           label=f"Media: {df_sent['char_length'].mean():.0f} caracteres")
ax.set_xlabel('Longitud del tweet (caracteres)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de la longitud de los tweets institucionales (2024–2026)')
ax.legend()
plt.tight_layout()
plt.savefig('histograma_longitud.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado: histograma_longitud.png")

# ── Nube de palabras ──────────────────────────────────────────────────────────
texto = ' '.join(df_sent['text'].astype(str))
texto = re.sub(r'http\S+', '', texto)
texto = re.sub(r'@\w+',   '', texto)
texto = re.sub(r'[^\w\sáéíóúñÁÉÍÓÚÑ]', ' ', texto)

stopwords_es = {
    'de','la','el','en','y','a','los','las','del','un','una','que','por',
    'con','para','su','al','se','es','lo','como','más','este','esta',
    'gob','minecogob','rt'
}

wc = WordCloud(
    width=1200, height=600, background_color='white',
    stopwords=stopwords_es, colormap='Blues', max_words=80
).generate(texto.lower())

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Nube de palabras del discurso institucional (2024–2026)', fontsize=14)
plt.tight_layout()
plt.savefig('nube_palabras.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado: nube_palabras.png")

## 11. Análisis estadístico: Correlaciones y Prueba de Raíz Unitaria (ADF)

**Correlación de Pearson** — mide la relación lineal entre las dos series;
reporta coeficiente y p-value para evaluar significancia estadística.

**Correlación de Spearman** — alternativa no paramétrica, robusta ante
distribuciones no normales; igualmente reporta coeficiente y p-value.

**Test de Dickey-Fuller Aumentado (ADF)** — verifica si cada serie es estacionaria,
requisito previo para los modelos de causalidad.
- H₀: la serie tiene raíz unitaria (no estacionaria).
- Si p < 0.05 → se rechaza H₀ → la serie es estacionaria.

In [ ]:
from scipy import stats
from statsmodels.tsa.stattools import adfuller

# ── Correlaciones ─────────────────────────────────────────────────────────────
pearson_r,  pearson_p  = stats.pearsonr( merged['sentiment_score'], merged['cci_value'])
spearman_r, spearman_p = stats.spearmanr(merged['sentiment_score'], merged['cci_value'])

print("=" * 55)
print("CORRELACIONES")
print("=" * 55)
print(f"Pearson   r={pearson_r:+.4f}  p={pearson_p:.4f}  "
      f"{'✅ Significativa' if pearson_p  < 0.05 else '⚠️  No significativa (p≥0.05)'}")
print(f"Spearman  r={spearman_r:+.4f}  p={spearman_p:.4f}  "
      f"{'✅ Significativa' if spearman_p < 0.05 else '⚠️  No significativa (p≥0.05)'}")

# ── Test ADF ──────────────────────────────────────────────────────────────────
def test_adf(serie, nombre):
    result       = adfuller(serie.dropna())
    estacionaria = result[1] < 0.05
    print(f"\nADF — {nombre}")
    print(f"  Estadístico : {result[0]:.4f}")
    print(f"  p-value     : {result[1]:.4f}")
    print(f"  Resultado   : {'✅ Estacionaria (p<0.05)' if estacionaria else '❌ No estacionaria → diferenciación necesaria'}")
    return estacionaria

print("\n" + "=" * 55)
print("ESTACIONARIEDAD (ADF)")
print("=" * 55)
stat_sent = test_adf(merged['sentiment_score'], 'Sentiment score')
stat_cci  = test_adf(merged['cci_value'],        'CCI España')

## 12. Causalidad de Granger y CCF sobre series en nivel

El CCI generalmente no es estacionario (contiene tendencia); se diferencia una vez
(`cci_diff`) antes de los tests de causalidad.

**Correlación Cruzada (CCF)** — calculada sobre `cci_diff` y computada
**dinámicamente** a partir de los datos reales. Identifica en qué desfases
temporales el sentimiento institucional anticipa variaciones en el CCI.

**Test de Causalidad de Granger** — evalúa formalmente si el sentiment score tiene
capacidad predictiva sobre `cci_diff` para **lags 1 a 4**.

> ⚠️ La causalidad de Granger mide **precedencia temporal predictiva**,
> no causalidad estructural.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests, ccf
import numpy as np

# ── Diferenciación del CCI ────────────────────────────────────────────────────
merged['cci_diff'] = merged['cci_value'].diff()

print("── Estacionariedad del CCI diferenciado ─────────────────────────────────")
test_adf(merged['cci_diff'].dropna(), 'CCI diferenciado (Δcci)')

# ── CCF dinámica sobre series estacionarias ───────────────────────────────────
serie_sent = merged['sentiment_score'].dropna().values
serie_cci  = merged['cci_diff'].dropna().values
n_comun    = min(len(serie_sent), len(serie_cci))
serie_sent = serie_sent[-n_comun:]
serie_cci  = serie_cci[-n_comun:]

ccf_values = ccf(serie_sent, serie_cci, nlags=6, adjusted=False)
ic_95      = 1.96 / np.sqrt(n_comun)

print("\n" + "=" * 55)
print("CORRELACIÓN CRUZADA (CCF) — Sentiment → Δcci")
print("=" * 55)
for lag, val in enumerate(ccf_values):
    marca = " ◀ supera IC 95%" if abs(val) > ic_95 else ""
    print(f"  Lag {lag}: {val:+.4f}{marca}")
print(f"  IC 95% = ±{ic_95:.4f}")

# ── Test de Granger (lags 1–4) ────────────────────────────────────────────────
print("\n" + "=" * 55)
print("CAUSALIDAD DE GRANGER — Sentiment → Δcci (lags 1–4)")
print("=" * 55)
datos_granger = merged[['cci_diff', 'sentiment_score']].dropna()
grangercausalitytests(datos_granger, maxlag=4, verbose=True)

In [ ]:
import matplotlib.pyplot as plt

# ── CCF bar chart ─────────────────────────────────────────────────────────────
lags    = list(range(len(ccf_values)))
colores = ['#2196F3' if v >= 0 else '#F44336' for v in ccf_values]

plt.figure(figsize=(10, 5))
plt.bar(lags, ccf_values, color=colores, alpha=0.85)
plt.axhline(y=0,       color='black', linewidth=0.8)
plt.axhline(y= ic_95,  color='gray',  linestyle='--', label=f'IC 95% (±{ic_95:.3f})')
plt.axhline(y=-ic_95,  color='gray',  linestyle='--')
plt.xlabel('Lag (meses)')
plt.ylabel('Correlación cruzada')
plt.title('CCF: Sentiment score → ΔCCI España')
plt.legend()
plt.tight_layout()
plt.savefig('ccf_plot.png', dpi=150)
plt.show()
print("✅ Guardado: ccf_plot.png")

# ── Exportar dataset consolidado ──────────────────────────────────────────────
merged.to_csv("dataset_final_tfm.csv", index=False)
print("✅ Guardado: dataset_final_tfm.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Gráfico 1: Histograma del sentiment score mensual ────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(merged['sentiment_score'], bins=12, color='#2196F3',
        edgecolor='white', alpha=0.85)
ax.axvline(merged['sentiment_score'].mean(), color='#F44336',
           linestyle='--', linewidth=2,
           label=f"Media: {merged['sentiment_score'].mean():.3f}")
ax.set_xlabel('Sentiment score mensual')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del sentiment score institucional (2024–2026)')
ax.legend()
plt.tight_layout()
plt.savefig('histograma_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Gráfico 2: Scatterplot CCI vs. Sentiment score ───────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merged['sentiment_score'], merged['cci_value'],
           color='#2196F3', alpha=0.7, s=80, edgecolors='white', linewidth=0.5)

z     = np.polyfit(merged['sentiment_score'], merged['cci_value'], 1)
p_fit = np.poly1d(z)
x_ln  = np.linspace(merged['sentiment_score'].min(), merged['sentiment_score'].max(), 100)
ax.plot(x_ln, p_fit(x_ln), color='#F44336', linestyle='--',
        linewidth=1.5, label=f'Tendencia (r={pearson_r:.3f})')

for _, row in merged.iterrows():
    ax.annotate(str(row['year_month']),
                (row['sentiment_score'], row['cci_value']),
                fontsize=7, alpha=0.6, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points')

ax.set_xlabel('Sentiment score institucional')
ax.set_ylabel('CCI España (ajustado estacionalmente)')
ax.set_title('Relación entre sentiment score institucional y CCI España (2024–2026)')
ax.legend()
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('scatterplot_cci_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Gráfico 3: Heatmap de correlaciones ──────────────────────────────────────
corr_data = merged[['sentiment_score', 'cci_value',
                     'mean_pos', 'mean_neg', 'mean_neu', 'n_tweets']].copy()
corr_data.columns = ['Sentiment score', 'CCI',
                     'Prob. positivo', 'Prob. negativo',
                     'Prob. neutro', 'N tweets']

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_data.corr(), annot=True, fmt='.3f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Matriz de correlaciones entre variables del análisis')
plt.tight_layout()
plt.savefig('heatmap_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Tres gráficos guardados correctamente")

## 13. Análisis de sensibilidad: series diferenciadas (Δ)

Verificación adicional aplicando la diferenciación a **ambas** series antes del
análisis CCF y Granger. Este enfoque garantiza la estacionariedad de los inputs
y refuerza la robustez de los resultados presentados en la memoria del TFM.

Pasos:
1. Diferenciar tanto `sentiment_score` como `cci_value` (ambas en Δ).
2. Re-testear estacionariedad sobre las series diferenciadas.
3. Calcular CCF y Granger sobre Δsent → Δcci.

In [ ]:
from statsmodels.tsa.stattools import adfuller, ccf, grangercausalitytests
import numpy as np

# ── 1. Diferenciar ambas series ───────────────────────────────────────────────
merged['sentiment_diff'] = merged['sentiment_score'].diff()
merged['cci_diff']       = merged['cci_value'].diff()

# ── 2. Re-testear estacionariedad (lag fijo = 1 para consistencia) ────────────
def test_adf_fijo(serie, nombre, lags=1):
    result = adfuller(serie.dropna(), maxlag=lags, autolag=None)
    print(f"ADF — {nombre} (lags={lags})")
    print(f"  Estadístico: {result[0]:.4f}  p-value: {result[1]:.4f}  "
          f"Estacionaria: {'Sí ✅' if result[1] < 0.05 else 'No ❌'}")
    return result

print("=" * 60)
print("ESTACIONARIEDAD TRAS DIFERENCIACIÓN")
print("=" * 60)
test_adf_fijo(merged['sentiment_diff'], 'Δsentiment score')
test_adf_fijo(merged['cci_diff'],       'ΔCCI España')

# ── 3. CCF sobre ambas series diferenciadas ───────────────────────────────────
serie_sdiff = merged['sentiment_diff'].dropna().values
serie_cdiff = merged['cci_diff'].dropna().values
n_comun     = min(len(serie_sdiff), len(serie_cdiff))
serie_sdiff = serie_sdiff[-n_comun:]
serie_cdiff = serie_cdiff[-n_comun:]

ccf_diff   = ccf(serie_sdiff, serie_cdiff, nlags=6, adjusted=False)
ic_95_diff = 1.96 / np.sqrt(n_comun)

print("\n" + "=" * 60)
print("CCF — Δsentiment → ΔCCI (series diferenciadas)")
print("=" * 60)
for lag, val in enumerate(ccf_diff):
    marca = " ◀ supera IC 95%" if abs(val) > ic_95_diff else ""
    print(f"  Lag {lag}: {val:+.4f}{marca}")
print(f"  IC 95% = ±{ic_95_diff:.4f}")

# ── 4. Granger sobre ambas series diferenciadas ───────────────────────────────
print("\n" + "=" * 60)
print("GRANGER — Δsentiment → ΔCCI (lags 1–3)")
print("=" * 60)
datos_granger_diff = merged[['cci_diff', 'sentiment_diff']].dropna()
grangercausalitytests(datos_granger_diff, maxlag=3, verbose=True)

In [ ]:
import matplotlib.pyplot as plt

# CCF plot — series diferenciadas (valores calculados dinámicamente arriba)
lags_d    = list(range(len(ccf_diff)))
colores_d = ['#2196F3' if v >= 0 else '#F44336' for v in ccf_diff]

plt.figure(figsize=(10, 5))
plt.bar(lags_d, ccf_diff, color=colores_d, alpha=0.85)
plt.axhline(y=0,            color='black', linewidth=0.8)
plt.axhline(y= ic_95_diff,  color='gray',  linestyle='--',
            label=f'IC 95% (±{ic_95_diff:.3f})')
plt.axhline(y=-ic_95_diff,  color='gray',  linestyle='--')
plt.xlabel('Lag (meses)')
plt.ylabel('Correlación cruzada')
plt.title('CCF: Δsentiment score → ΔCCI España (series diferenciadas)')
plt.legend()
plt.tight_layout()
plt.savefig('ccf_plot_diff.png', dpi=150)
plt.show()
print("✅ Guardado: ccf_plot_diff.png")

## 14. Distribución de clases de sentimiento

Visualización del balance de etiquetas en el corpus completo.
Los valores se obtienen directamente de `df_sent` para garantizar
la reproducibilidad (sin valores hardcodeados).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_sent = pd.read_csv("tweets_con_sentimiento.csv")

# ── Valores calculados dinámicamente desde df_sent ────────────────────────────
conteos    = df_sent['sentiment'].value_counts()
categorias = ['NEU', 'POS', 'NEG']
etiquetas  = ['Neutro', 'Positivo', 'Negativo']
valores    = [conteos.get(cat, 0) for cat in categorias]
colores    = ['#78909C', '#4CAF50', '#F44336']
total      = sum(valores)

fig, ax = plt.subplots(figsize=(7, 5))
barras = ax.bar(etiquetas, valores, color=colores, edgecolor='black', linewidth=0.8)

for barra, val in zip(barras, valores):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 8,
        f"{val}\n({val / total * 100:.1f}%)",
        ha='center', fontsize=10
    )

ax.set_ylabel('Número de tweets')
ax.set_title('Distribución de clases de sentimiento (@_minecogob, 2024–2026)')
ax.set_ylim(0, max(valores) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('distribucion_sentimiento.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Guardado: distribucion_sentimiento.png")

---

## Nota final de consistencia con la memoria del TFM

Los resultados generados por este notebook sustentan las cifras reportadas en la
memoria del TFM:

- **1.100 publicaciones** extraídas del perfil `@_minecogob` vía API v2 (periodo 2024–2026).
- **716 tweets originales** tras excluir retweets (verificado con `assert`).
- Predominancia de **sentimiento neutro** en el discurso institucional.
- **Correlaciones contemporáneas débiles y no significativas** entre sentiment score y CCI.
- Patrón exploratorio de **desfase inverso** en la CCF (lags 3–5).
- **Ausencia de causalidad de Granger estadísticamente significativa** en lags 1–4.

Este notebook es la evidencia técnica reproducible del procedimiento descrito en la
Sección de Metodología del TFM. Puede ejecutarse en su totalidad a partir de los CSV
originales sin necesidad de reactivar la conexión a la API de X/Twitter.

**Archivos generados por este pipeline:**

| Archivo | Descripción |
|---|---|
| `cci_spain.csv` | CCI España ajustado estacionalmente (Eurostat) |
| `tweets_minecogob_limpio.csv` | Corpus limpio con columna `text_clean` |
| `tweets_con_sentimiento.csv` | Corpus con etiquetas y probabilidades |
| `dataset_final_tfm.csv` | Dataset consolidado (series mensuales fusionadas) |
| `series_temporales.png` | Evolución comparada de ambas series |
| `ccf_plot.png` | CCF: sentiment score → ΔCCI |
| `ccf_plot_diff.png` | CCF: Δsentiment → ΔCCI (series diferenciadas) |
| `histograma_sentiment.png` | Distribución del score mensual |
| `histograma_longitud.png` | Longitud de tweets |
| `scatterplot_cci_sentiment.png` | Diagrama de dispersión |
| `heatmap_correlaciones.png` | Matriz de correlaciones |
| `distribucion_sentimiento.png` | Distribución POS/NEG/NEU |
| `nube_palabras.png` | Word cloud del corpus |